# 07 - Contratos y schemas con Pydantic

Objetivo: validar inputs antes de tocar GitHub.


## Grafico guia

```mermaid
flowchart TD
    A["Input del LLM"] --> B["Schema Pydantic"]
    B -->|"valido"| C["Ejecutar tool"]
    B -->|"invalido"| D["Error explicito"]
    C --> E["Output con contrato"]
```


In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class CreateRepoInput(BaseModel):
    name: str = Field(..., min_length=3)
    description: str = ""
    private: bool = True

class UpsertFileInput(BaseModel):
    owner: str = Field(..., min_length=1)
    repo: str = Field(..., min_length=1)
    path: str = Field(..., min_length=1)
    content: str
    commit_message: str = Field(..., min_length=5)
    branch: str = "main"

class ToolOutput(BaseModel):
    status: Literal["planned", "created", "updated"]
    detail: str


In [ ]:
def validate_create_repo(args):
    try:
        parsed = CreateRepoInput.model_validate(args)
    except ValidationError as exc:
        return {"error": "validation_error", "details": exc.errors()}
    return ToolOutput(status="planned", detail=f"Repo {parsed.name} validado").model_dump()

validate_create_repo({"name": "aem4-demo", "description": "demo"})


In [ ]:
validate_create_repo({"name": "x"})


## Lectura docente

La validacion evita que el server ejecute side effects con payloads incompletos. En el MCP real, los `Field(...)` de Pydantic documentan y restringen los argumentos.
